In [22]:
import pandas as pd
import numpy as np
import os

In [23]:
pasta_dados = r"C:\Users\Joao.Maria\Documents\ProjetoAutomacao-FDEVS\data\input"
pasta_saida_dados = r"C:\Users\Joao.Maria\Documents\ProjetoAutomacao-FDEVS\data\output"

dados_campanha = os.path.join(pasta_dados, "Campanha Incentivo - Distribuição Vinho.xlsx")
dados_configuracao_campanha = os.path.join(pasta_dados, "Configuracao da Campanha.xlsx")
dados_bruto = os.path.join(pasta_dados, "Dados_Brutos.csv")

In [24]:
relatorio_campanha_premiacao = pd.read_excel(dados_campanha, sheet_name="Premiação Distribuidores")
relatorio_campanha_detalhe = pd.read_excel(dados_campanha, sheet_name="Detalhamento Mês Apurado")
configuracao_campanha_cliente = pd.read_excel(dados_configuracao_campanha, sheet_name="Cliente")
configuracao_campanha_produto = pd.read_excel(dados_configuracao_campanha, sheet_name="Produto")
relatorio_bruto = pd.read_csv(dados_bruto, sep=";", encoding="utf-8", dtype={"NUMERODOCUMENTO": "str"})

In [25]:
dados_bruto = (
    relatorio_bruto[relatorio_bruto["CODIGOCLIENTE"].isin(
        configuracao_campanha_cliente["CODIGOCLIENTE"]
    )]
)

dados_bruto = (
    dados_bruto[dados_bruto["CODIGOPRODUTO"].isin(
        configuracao_campanha_produto["CODIGOPRODUTO"]
    )]
)

CFOP_CAMPANHA = [
    5102, 5106, 5110, 6102, 6106, 6110, 5160, 6160, 6910, 5910
]

dados_bruto = dados_bruto[dados_bruto["CFOP"].isin(CFOP_CAMPANHA)]

dados_bruto = dados_bruto.rename(columns={
    "VOLUME": "VOLUME BRUTO"
})

### 1. Validação Base Apurada x Base Bruta

In [26]:
validacao_volume_campanha_bruto = pd.merge(
    relatorio_campanha_detalhe,
    dados_bruto[["CODIGOCLIENTE", "CODIGODOCUMENTO", "CODIGOPRODUTO", "VOLUME BRUTO"]],
    how="left",
    on=["CODIGOCLIENTE", "CODIGODOCUMENTO", "CODIGOPRODUTO"]
)

validacao_volume_campanha_bruto["Validação"] = np.where(
    validacao_volume_campanha_bruto["VOLUME"] != validacao_volume_campanha_bruto["VOLUME BRUTO"],
    "Divergente",
    "Correto"
)

validacao_volume_campanha_bruto_divergente = validacao_volume_campanha_bruto[validacao_volume_campanha_bruto["Validação"] != "Correto"]

validacao_volume_campanha_bruto_divergente

,CODIGOCAMPANHA,DESCRICAOCAMPANHA,CODIGOCLIENTE,NOMECLIENTE,CODIGOGRUPOCLIENTE,CODIGODOCUMENTO,NUMERODOCUMENTO,NUMEROSERIE,DATAEMISSAO,CODIGOFAMILIAPRODUTO,...,NOMEGRUPOPRODUTO,CODIGOPRODUTO,NOMEPRODUTO,CODIGOCFOP,DESCRICAOTIPOFATURAMENTO,FATORPRODUTO,VOLUME,DATAAPURACAO,VOLUME BRUTO,Validação
14,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5729295,17348,1,2026-02-20,80000007,...,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,1202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,Divergente
15,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5729295,17348,1,2026-02-20,80000007,...,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,1202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,Divergente
114,4002,Harmonização Master - Vinho Tintos Importados ...,10004204,LOGÍSTICA VALE DAS BEBIDAS,2,5728490,136345,1,2026-02-20,80000007,...,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,5110,VENDA,1,25,2026-03-09 13:36:58.995,200.0,Divergente
183,4002,Harmonização Master - Vinho Tintos Importados ...,10004219,SUPRIMENTOS MASTER BEBIDAS,2,5762186,176536,1,2026-02-19,80000007,...,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,1202,DEVOLUÇÃO DE VENDA,1,-10,2026-03-09 13:36:58.995,NaN,Divergente
346,4002,Harmonização Master - Vinho Tintos Importados ...,10004227,LOGÍSTICA INTEGRAL DE BEBIDAS,2,5744799,44502,1,2026-02-24,80000007,...,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,2202,DEVOLUÇÃO DE VENDA,1,-50,2026-03-09 13:36:58.995,NaN,Divergente
559,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5703114,67183,1,2026-02-13,80000007,...,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,1202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,Divergente
560,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5757643,36677,1,2026-02-28,80000007,...,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,2202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,Divergente


### 2. Validação Premiação

In [27]:
def percentual_premiacao_crescimento(crescimento):
    
    if crescimento >= 0.20: 
        return 0.10
    
    if crescimento >= 0.15: 
        return 0.08
    
    if crescimento >= 0.10: 
        return 0.06
    
    if crescimento >= 0.05: 
        return 0.03
    
    return 0.0


def validar_multiplas_regras(dataframe, regras, separador= " | "):

    """
    Aplica múltiplas regras vetorizadas e retorna uma única coluna:
    - "Correto" se nenhuma regra falhar

    Args:
        dataframe (pd.DataFrame): DataFrame geral de validação
        regras (list -> tuple): Lista com tuplas dentro com validações
        separador (str): Caracter que separa as divergências
    """

    mensagens = np.full(len(dataframe), "", dtype=object)

    for mascara, texto in regras:
        mensagens = np.where(
            mascara,
            np.where(
                mensagens == "",
                texto,
                mensagens + separador + texto
            ),
            mensagens
        )

    return np.where(mensagens == "", "Correto", mensagens)

In [28]:
campanha_detalhe = (
    relatorio_campanha_detalhe.groupby(["CODIGOCLIENTE", "CODIGOPRODUTO"])
    ["VOLUME"].sum()
    .reset_index()
)

validacao_preamiacao = pd.merge(
    relatorio_campanha_premiacao,
    campanha_detalhe,
    how="left",
    on=["CODIGOCLIENTE", "CODIGOPRODUTO"]
)

validacao_preamiacao["% Crescimento Calculado"] = (
    (validacao_preamiacao["VOLUMEREALIZADO"] / validacao_preamiacao["METAVOLUMEVENDA"]) - 1
).round(4)

validacao_preamiacao["% Premiação Calculado"] = validacao_preamiacao["% Crescimento Calculado"].apply(percentual_premiacao_crescimento)

validacao_preamiacao["Valor Premiacao Calculado"] = (
    validacao_preamiacao["VALORVENDAPREMIACAO"] * validacao_preamiacao["% Premiação Calculado"]
)

validacao_preamiacao["Validação"] = validar_multiplas_regras(
    validacao_preamiacao,
    regras=[
        (
            validacao_preamiacao["% Premiação Calculado"] != validacao_preamiacao["PERCENTUALPREMIACAO"], 
            "Percentual Premiação divergente do Cálculo"
        ),
        (
            validacao_preamiacao["VALORPREMIACAO"] != validacao_preamiacao["Valor Premiacao Calculado"], 
            "Valor Premiação divergente do Cálculo"
        ),
    ]
)

In [30]:
with pd.ExcelWriter(os.path.join(pasta_saida_dados, "Indicadores.xlsx"), engine="xlsxwriter") as writer:

    validacao_volume_campanha_bruto.to_excel(
        writer,
        sheet_name="Dados Apuração x Bruto",
        index=False
    )
    
    validacao_preamiacao.to_excel(
        writer,
        sheet_name="Premiação",
        index=False
    )